# Webcam Recorder Widget

Records webcam **video + audio** to disk, with live filters baked in.

Run the next cell, then click **Start Camera** → **Record** → **Stop**. The clip is saved automatically.

In [9]:
# Bootstrap: ensure the widget's deps exist in *this* Python (e.g. Blender's
# bundled interpreter, which starts bare). Safe to re-run — pip skips anything
# already satisfied. imageio-ffmpeg ships a static ffmpeg binary so to_mp4()
# works without a system ffmpeg install.
import importlib.util
import subprocess
import sys

_required = {
    "anywidget": "anywidget>=0.9.0",
    "ipywidgets": "ipywidgets>=8.0",
    "traitlets": "traitlets>=5.0",
    "imageio_ffmpeg": "imageio-ffmpeg>=0.4.9",
}
_missing = [spec for mod, spec in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("Installed:", ", ".join(_missing))
else:
    print("All widget dependencies already present.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 18.7 MB/s  0:00:01 eta 0:00:01
Installed: imageio-ffmpeg>=0.4.9


In [10]:
# Running inside Blender's bundled Python, the package isn't pip-installed, so
# make it importable from this project's `src/` via an absolute path.
import sys

PROJECT_DIR = "/Users/jan-hendrik/projects/webcam-recorder-widget"
if f"{PROJECT_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{PROJECT_DIR}/src")

from webcam_recorder import WebcamRecorderWidget

# Absolute save_dir too: Blender's working directory is not this project, so a
# relative path would drop recordings wherever Blender was launched from.
w = WebcamRecorderWidget(
    save_dir=f"{PROJECT_DIR}/recordings",
    filename_prefix="take",
)
w

In [11]:
# Where did the last clip land?
w.last_saved_path

''

## Drive it from Python (timed capture)

In [12]:
import time

w.record()          # start camera (if needed) + recording


In [13]:
w.stop_recording()  # flush to disk
print(w.last_saved_path)

In [14]:
print(w.last_saved_path)

/Users/jan-hendrik/projects/webcam-recorder-widget/recordings/take_20260601_174441.webm


## Post-processing (optional, requires ffmpeg)

In [17]:
w.to_mp4()          # -> H.264/AAC .mp4 next to the .webm
# w.extract_audio() # -> standalone .m4a

RuntimeError: ffmpeg not found on PATH. Install it (e.g. `brew install ffmpeg`).

## Send the recording into Blender's Video Sequence Editor

Run this **inside Blender's Python**. It saves the last clip as an mp4 in your Downloads folder, then drops it into the next free channel of the VSE.

In [8]:
import pathlib


def add_recording_to_vse(widget, directory=None, *, with_audio=True, frame_start=None):
    """Save the widget's latest recording as mp4, then add it to Blender's VSE.

    1. Transcodes the last clip to H.264/AAC mp4 in `directory`
       (default: your ~/Downloads folder).
    2. Adds it as a movie strip in the next free channel of the Video Sequence
       Editor (its audio goes on the channel above when `with_audio`).

    `frame_start` defaults to the scene's current frame. Returns the movie strip.
    Run this inside Blender's bundled Python, where `bpy` is available.
    """
    try:
        import bpy
    except ImportError as exc:
        print("nope")
        raise RuntimeError(
            "bpy not available — run this notebook inside Blender's Python."
        ) from exc

    target_dir = (
        pathlib.Path(directory).expanduser()
        if directory
        else pathlib.Path.home() / "Downloads"
    )
    mp4_path = pathlib.Path(widget.to_mp4(dest=target_dir))

    scene = bpy.context.scene
    if not scene.sequence_editor:
        scene.sequence_editor_create()
    sed = scene.sequence_editor

    # Blender 4.4 renamed `sequences*` -> `strips*`; support both.
    all_strips = getattr(sed, "strips_all", None)
    if all_strips is None:
        all_strips = getattr(sed, "sequences_all", [])
    channel = max((s.channel for s in all_strips), default=0) + 1
    start = scene.frame_current if frame_start is None else int(frame_start)

    container = getattr(sed, "strips", None) or sed.sequences
    movie = container.new_movie(
        name=mp4_path.stem, filepath=str(mp4_path), channel=channel, frame_start=start
    )
    if with_audio:
        container.new_sound(
            name=f"{mp4_path.stem}_audio",
            filepath=str(mp4_path),
            channel=channel + 1,
            frame_start=start,
        )
    print(f"Added '{mp4_path.name}' to VSE channel {channel} at frame {start}.")
    return movie


# Record a clip first (Start Camera -> Record -> Stop, or w.record()/w.stop_recording()),
# then drop it straight into the sequencer:
add_recording_to_vse(w)

RuntimeError: ffmpeg not found on PATH. Install it (e.g. `brew install ffmpeg`).